### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 1 - Absorpcja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity

Wyniki dla STL:

In [3]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score
from rdkit.Chem import Descriptors, Lipinski

In [7]:
class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]
        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column] if self.y_column in df.columns else np.nan
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                desc_vector = [
                    Descriptors.MolWt(mol),
                    Descriptors.MolLogP(mol),
                    Lipinski.NumHDonors(mol),
                    Lipinski.NumHAcceptors(mol),
                    Descriptors.TPSA(mol),
                    Lipinski.NumRotatableBonds(mol),
                    Lipinski.NumAromaticRings(mol),
                    Descriptors.HeavyAtomCount(mol),
                    Descriptors.MolMR(mol),
                    Lipinski.FractionCSP3(mol)
                ]
            else:
                # Wstawiamy wektor zer o długości 10, jeśli cząsteczka jest błędna
                desc_vector = [0.0] * len(self.feature_names)

            features.append(desc_vector)
            labels.append(y)

        features_array = np.array(features)

        if fit_scaler:
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels).reshape(-1, 1)

In [8]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [9]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [10]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    # Bierzemy wymiar wejściowy z liczby kolumn X_train_np
    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [11]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')


    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [12]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0


    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')


    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [13]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [14]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [15]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer, fit_scaler=True):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    # TUTAJ JEST ZMIANA: dodajemy przekazanie parametru fit_scaler
    X_features, _ = featurizer(df_temp, fit_scaler=fit_scaler)

    # Sprawdzenie czy X zawiera NaN
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

# Test1: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca)

Test pierwszy sprawdzający czy klirens wątrobowy (Clearance Hepatocyte) pomoże w predykcji okresu półtrwania (Half Life) - oba zadania są bezpośrednio powiązane mechanistycznie. Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [16]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')

# 2. Przygotowanie danych MTL
# Na danych TRENINGOWYCH dopasowujemy skaler (fit_scaler=True)
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear], reg_tasks, featurizer, fit_scaler=True)

# Na danych TESTOWYCH wykorzystujemy tylko dopasowany wcześniej skaler (fit_scaler=False)
X_test_mtl, y_test_dict = prepare_mtl_data_final([test_halflife, test_clear], reg_tasks, featurizer, fit_scaler=False)

loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.3158
  Epoka  20 | Total Loss: 1.7514
  Epoka  40 | Total Loss: 1.5544
  Epoka  60 | Total Loss: 1.3432
  Epoka  80 | Total Loss: 1.2098

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 100.7277
  MAE      : 29.8412
  R²       : 0.2927


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 47.8512
  MAE      : 35.3043
  R²       : 0.0996


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.4362
  Epoka  20 | Total Loss: 0.8417
  Epoka  40 | Total Loss: 0.7364
  Epoka  60 | Total Loss: 0.6716
  Epoka  80 | Total Loss: 0.5672

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 102.0111
  MAE      : 30.2343
  R²   

# Test2: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik.

In [19]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')


# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_cyp], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_cyp],  reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.4812
  Epoka  20 | Total Loss: 2.3101
  Epoka  40 | Total Loss: 2.0715
  Epoka  60 | Total Loss: 1.9051
  Epoka  80 | Total Loss: 1.7811

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 100.7354
  MAE      : 31.4691
  R²       : 0.2926


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 47.8074
  MAE      : 34.4168
  R²       : 0.1013


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7210
  F1       : 0.6647
  AUROC    : 0.8103


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0631
  Epoka  20 | Total Loss: 0.7732
  Epoka  40 | Total Loss: 0.6904
  Epoka  60 | Total Loss: 0.6485
  Epoka  80 | Total Loss: 0.5998

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Un

# Test3: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo)

Dokładamy VDss - kompletny zestaw eliminacyjny (4 endpointy biologicznie powiązane). Sprawdzamy czy pełne pokrycie domeny daje najlepsze wyniki, czy w pewnym momencie wyniki przestają się poprawiać.

In [20]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_vdss, train_cyp], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_vdss, test_cyp],  reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.1869
  Epoka  20 | Total Loss: 3.0544
  Epoka  40 | Total Loss: 2.8398
  Epoka  60 | Total Loss: 2.6372
  Epoka  80 | Total Loss: 2.2996

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 98.6283
  MAE      : 31.6917
  R²       : 0.3219


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 47.8039
  MAE      : 34.7616
  R²       : 0.1014


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 12.3961
  MAE      : 5.8705
  R²       : -2.2803


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7202
  F1       : 0.6702
  AUROC    : 0.8089


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.2130
  Epoka  20 | Total Loss: 0.8119
  Epoka  

# Test4: Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith)

Sprawdzamy parę regresja + klasyfikacja - oba zadania bezpośrednio związane z metabolizmem wątrobowym. Jednocześnie porównujemy STL vs MTL.

In [21]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Clearance_Hepatocyte_AZ']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_clear, train_cyp], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_clear, test_cyp],  reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8222
  Epoka  20 | Total Loss: 1.3590
  Epoka  40 | Total Loss: 1.2651
  Epoka  60 | Total Loss: 1.2216
  Epoka  80 | Total Loss: 1.1547

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 48.4855
  MAE      : 35.4019
  R²       : 0.0756


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7344
  F1       : 0.6791
  AUROC    : 0.8130


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0351
  Epoka  20 | Total Loss: 0.7047
  Epoka  40 | Total Loss: 0.6446
  Epoka  60 | Total Loss: 0.6105
  Epoka  80 | Total Loss: 0.5861

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Uniform Average
  RMSE     : 48.3459
  MAE      : 34.6901
  

# Test5: Half Life (Obach) + CYP3A4 Inhibition (Veith)

Sprawdzamy czy CYP3A4 (klasyfikacja) pomoże regresji Half Life - są to zadania powiązane pośrednio przez metabolizm enzymu CYP3A4.

In [22]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach']
class_tasks = ['CYP3A4_Veith'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_cyp], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_cyp],  reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8986
  Epoka  20 | Total Loss: 1.4322
  Epoka  40 | Total Loss: 1.2848
  Epoka  60 | Total Loss: 1.2140
  Epoka  80 | Total Loss: 1.0866

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 99.3829
  MAE      : 31.0894
  R²       : 0.3114


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7303
  F1       : 0.6767
  AUROC    : 0.8135


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9068
  Epoka  20 | Total Loss: 0.6701
  Epoka  40 | Total Loss: 0.6240
  Epoka  60 | Total Loss: 0.5296
  Epoka  80 | Total Loss: 0.4389

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 103.3661
  MAE      : 33.2408
  R²       : 0.25

# Test6: Half Life (Obach) + AMES Mutagenicity

Sprawdzamy łączenie z niepowiązanym biologicznie zadaniem (AMES - mutagenność). Oczekujemy transferu negatywnego.

In [23]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ładowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_ames], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_ames],  reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.8091
  Epoka  20 | Total Loss: 1.5669
  Epoka  40 | Total Loss: 1.4568
  Epoka  60 | Total Loss: 1.4179
  Epoka  80 | Total Loss: 1.3700

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 109.0168
  MAE      : 33.1521
  R²       : 0.1715


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7187
  F1       : 0.7349
  AUROC    : 0.7838


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9799
  Epoka  20 | Total Loss: 0.7607
  Epoka  40 | Total Loss: 0.6961
  Epoka  60 | Total Loss: 0.6607
  Epoka  80 | Total Loss: 0.6109

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Uniform Average
  RMSE     : 103.6273
  MAE      : 32.8195
  R²       : 0.2514


Wy

# Test7: Half Life (Obach) + Clearance Hepatocyte (AstraZeneca) + CYP3A4 Inhibition (Veith) + VDss (Lombardo) + AMES Mutagenicity

In [24]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Half_Life_Obach', 'Clearance_Hepatocyte_AZ', 'VDss_Lombardo']
class_tasks = ['CYP3A4_Veith', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_eliminacja_descriptors.txt"

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# 1. Ladowanie danych
train_halflife, test_halflife = load_split_pickle('Half_Life_Obach')
train_clear, test_clear = load_split_pickle('Clearance_Hepatocyte_AZ')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_cyp, test_cyp = load_split_pickle('CYP3A4_Veith')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_halflife, train_clear, train_vdss, train_cyp, train_ames], reg_tasks + class_tasks, featurizer, fit_scaler=True)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_halflife, test_clear, test_vdss, test_cyp, test_ames], reg_tasks + class_tasks, featurizer, fit_scaler=False)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 5.0347
  Epoka  20 | Total Loss: 3.9048
  Epoka  40 | Total Loss: 3.5695
  Epoka  60 | Total Loss: 3.3504
  Epoka  80 | Total Loss: 3.0279

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Half_Life_Obach

  Loss Weighting: Weighted Loss Sum
  RMSE     : 100.0279
  MAE      : 31.1714
  R²       : 0.3025


Wyniki dla zadania (Regresja): Clearance_Hepatocyte_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 47.4118
  MAE      : 35.5974
  R²       : 0.1161


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 11.5898
  MAE      : 6.1827
  R²       : -1.8674


Wyniki dla zadania (Klasyfikacja): CYP3A4_Veith

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7275
  F1       : 0.6754
  AUROC    : 0.8098


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.6960
  F1       : 0.7192
  AUROC    